# Baseline Interpolation

In [1]:
# ============================================
# STEP 5.1 — Import Libraries & Define Paths
# ============================================

import pandas as pd
import numpy as np
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw data folder
RAW_DATA = PROJECT_ROOT / "Data" / "raw data"

# Processed data folder
PROCESSED_DATA = PROJECT_ROOT / "Data" / "processed"

# Dataset path
AIR_FILE = RAW_DATA / "air_quality_2019_2021_hourly.csv"
MET_FILE = RAW_DATA / "hourly_meteorology_16stations_2019_2021.csv"

# Fixed artificial mask created in STEP 3
MASK_FILE = PROCESSED_DATA / "fixed_artificial_mask_new.csv"

print("Project Root:")
print(PROJECT_ROOT)

print("\nAir Quality File:")
print(AIR_FILE)

print("\nMeteorology File:")
print(MET_FILE)

print("\nFixed Mask File:")
print(MASK_FILE)

Project Root:
e:\Edge_air_pollution imputation

Air Quality File:
e:\Edge_air_pollution imputation\Data\raw data\air_quality_2019_2021_hourly.csv

Meteorology File:
e:\Edge_air_pollution imputation\Data\raw data\hourly_meteorology_16stations_2019_2021.csv

Fixed Mask File:
e:\Edge_air_pollution imputation\Data\processed\fixed_artificial_mask_new.csv


In [2]:
# ============================================
# STEP 5.2 — Load Dataset & Fixed Mask
# ============================================

# Load raw datasets
air_df = pd.read_csv(AIR_FILE)
met_df = pd.read_csv(MET_FILE)

# Convert timestamp
air_df["timestamp"] = pd.to_datetime(air_df["timestamp"])
met_df["timestamp"] = pd.to_datetime(met_df["timestamp"])

# Merge Air Quality + Meteorology
merged_df = pd.merge(
    air_df,
    met_df,
    on=["station_id", "station_name", "timestamp"],
    how="inner"
)

# Load fixed artificial mask
fixed_mask = pd.read_csv(MASK_FILE)

print("Air Quality shape:", air_df.shape)
print("Meteorology shape:", met_df.shape)
print("Merged dataset shape:", merged_df.shape)
print("Fixed mask shape:", fixed_mask.shape)

Air Quality shape: (420864, 10)
Meteorology shape: (420864, 9)
Merged dataset shape: (420864, 16)
Fixed mask shape: (420864, 13)


In [3]:
# ============================================
# STEP 5.3 — Define & Verify 13 Features
# ============================================

pollution_features = [
    "pm25",
    "pm10",
    "no2",
    "o3",
    "so2",
    "nox",
    "co"
]

weather_features = [
    "temperature",
    "relative_humidity",
    "wind_speed",
    "wind_direction",
    "pressure",
    "rainfall"
]

feature_columns = pollution_features + weather_features

print("Total features:", len(feature_columns))

print("\nFeature columns:")
for i, feature in enumerate(feature_columns, start=1):
    print(f"{i:2}. {feature}")

print("\nMerged dataset feature columns present:",
      all(feature in merged_df.columns for feature in feature_columns))

print("Fixed mask columns match:",
      list(fixed_mask.columns) == feature_columns)

Total features: 13

Feature columns:
 1. pm25
 2. pm10
 3. no2
 4. o3
 5. so2
 6. nox
 7. co
 8. temperature
 9. relative_humidity
10. wind_speed
11. wind_direction
12. pressure
13. rainfall

Merged dataset feature columns present: True
Fixed mask columns match: True


In [4]:
# ============================================
# STEP 5.4 — Create Ground Truth & Masked Data
# ============================================

# Original data as ground truth
ground_truth_df = merged_df.copy()

# Copy for interpolation input
masked_df = merged_df.copy()

# Apply fixed artificial mask
# 1 = available, 0 = masked
for feature in feature_columns:
    masked_df.loc[fixed_mask[feature] == 0, feature] = np.nan

print("Ground truth shape:", ground_truth_df.shape)
print("Masked data shape:", masked_df.shape)

# Count NaN values in model features
print("\nNaN count in masked data:")
print(masked_df[feature_columns].isna().sum().sum())

Ground truth shape: (420864, 16)
Masked data shape: (420864, 16)

NaN count in masked data:
1321566


In [5]:
# ============================================
# STEP 5.5 — Sort Data Station-wise & Time-wise
# ============================================

# Sort both datasets using the same station and time order
ground_truth_df = ground_truth_df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

masked_df = masked_df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

print("Ground truth sorted:", ground_truth_df.shape)
print("Masked data sorted:", masked_df.shape)

# Verify first and last timestamps
print("\nFirst row:")
print(masked_df[["station_id", "timestamp"]].head(1))

print("\nLast row:")
print(masked_df[["station_id", "timestamp"]].tail(1))

Ground truth sorted: (420864, 16)
Masked data sorted: (420864, 16)

First row:
   station_id  timestamp
0          66 2019-01-01

Last row:
        station_id           timestamp
420863          83 2021-12-31 23:00:00


In [6]:
# ============================================
# STEP 5.6 — Station-wise Linear Interpolation
# ============================================

interpolated_df = masked_df.copy()

# Interpolate each feature separately for each station
for feature in feature_columns:
    interpolated_df[feature] = (
        interpolated_df
        .groupby("station_id")[feature]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit_direction="both"
            )
        )
    )

print("Interpolation completed.")

print("\nRemaining NaN values:")
print(interpolated_df[feature_columns].isna().sum().sum())

Interpolation completed.

Remaining NaN values:
210432


In [7]:
# ============================================
# STEP 5.7 — Verify Interpolated Values
# ============================================

# Artificially masked positions:
# Original mask = 1 and fixed mask = 0
artificial_mask_positions = (
    (merged_df[feature_columns].notna()) &
    (fixed_mask[feature_columns] == 0)
)

# Count how many artificial positions were created
total_artificial_positions = artificial_mask_positions.sum().sum()

# Count how many of those positions now have interpolated values
filled_artificial_positions = (
    interpolated_df[feature_columns]
    .notna() &
    artificial_mask_positions
).sum().sum()

# Count artificial positions still remaining NaN
remaining_artificial_nan = (
    interpolated_df[feature_columns].isna() &
    artificial_mask_positions
).sum().sum()

print("Total artificial masked positions:", total_artificial_positions)
print("Successfully interpolated:", filled_artificial_positions)
print("Still NaN:", remaining_artificial_nan)

Total artificial masked positions: 1037455
Successfully interpolated: 1011602
Still NaN: 25853


In [10]:
# ============================================
# STEP 5.8A — Align Fixed Mask
# ============================================

# Get the exact sorting order used for the dataset
sort_order = merged_df.sort_values(
    ["station_id", "timestamp"]
).index

# Reorder fixed mask using the same order
fixed_mask_sorted = (
    fixed_mask
    .loc[sort_order]
    .reset_index(drop=True)
)

# Verify alignment
print("Sorted mask shape:", fixed_mask_sorted.shape)
print("Ground truth shape:", ground_truth_df.shape)
print("Interpolated data shape:", interpolated_df.shape)

print(
    "\nMask and ground truth rows match:",
    len(fixed_mask_sorted) == len(ground_truth_df)
)

Sorted mask shape: (420864, 13)
Ground truth shape: (420864, 16)
Interpolated data shape: (420864, 16)

Mask and ground truth rows match: True


In [11]:
# ============================================
# STEP 5.8B — Calculate MAE & RMSE
# ============================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

results = []

for feature in feature_columns:

    # Artificially masked positions
    artificial_positions = (
        ground_truth_df[feature].notna() &
        (fixed_mask_sorted[feature] == 0)
    )

    # Interpolation must have produced a value
    interpolation_available = (
        interpolated_df[feature].notna()
    )

    # Final valid evaluation positions
    valid_positions = (
        artificial_positions &
        interpolation_available
    )

    # Ground truth
    y_true = ground_truth_df.loc[
        valid_positions, feature
    ]

    # Interpolated prediction
    y_pred = interpolated_df.loc[
        valid_positions, feature
    ]

    # Extra safety: remove any remaining NaN
    valid_numeric = (
        y_true.notna() &
        y_pred.notna()
    )

    y_true = y_true[valid_numeric]
    y_pred = y_pred[valid_numeric]

    # Metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    results.append({
        "Feature": feature,
        "Evaluation_Points": len(y_true),
        "MAE": mae,
        "RMSE": rmse
    })

# Create final results table
interpolation_results = pd.DataFrame(results)

print(interpolation_results.to_string(index=False))

          Feature  Evaluation_Points       MAE      RMSE
             pm25              81706  1.928010  2.811731
             pm10              82146  2.660454  3.965873
              no2              81488  5.481378  8.302123
               o3              82138  5.498488  8.463845
              so2              82128  0.524086  0.950927
              nox              76520 13.893732 26.335951
               co              46323  3.506766  5.954657
      temperature              84162  0.250113  0.373373
relative_humidity              84296  1.491802  2.309094
       wind_speed              84069  0.298734  0.435866
   wind_direction              84344 12.304220 37.624868
         pressure              83978  0.206277  0.274462
         rainfall              84157  0.167156  0.710222


In [12]:
# ============================================
# STEP 5.9 — Save Interpolation Baseline Results
# ============================================

# Output file path
INTERPOLATION_RESULTS_FILE = (
    PROCESSED_DATA / "interpolation_baseline_results_new.csv"
)

# Save results
interpolation_results.to_csv(
    INTERPOLATION_RESULTS_FILE,
    index=False
)

print("Interpolation baseline results saved successfully!")
print("\nFile:")
print(INTERPOLATION_RESULTS_FILE)

print("\nSaved results:")
print(interpolation_results)

Interpolation baseline results saved successfully!

File:
e:\Edge_air_pollution imputation\Data\processed\interpolation_baseline_results_new.csv

Saved results:
              Feature  Evaluation_Points        MAE       RMSE
0                pm25              81706   1.928010   2.811731
1                pm10              82146   2.660454   3.965873
2                 no2              81488   5.481378   8.302123
3                  o3              82138   5.498488   8.463845
4                 so2              82128   0.524086   0.950927
5                 nox              76520  13.893732  26.335951
6                  co              46323   3.506766   5.954657
7         temperature              84162   0.250113   0.373373
8   relative_humidity              84296   1.491802   2.309094
9          wind_speed              84069   0.298734   0.435866
10     wind_direction              84344  12.304220  37.624868
11           pressure              83978   0.206277   0.274462
12           rainfal